# DermaMNIST + HDFS + Spark + PyTorch DDP

In this notebook, we extend the Spark + HDFS DermaMNIST pipeline by using PyTorch Distributed Data Parallel (DDP) for CNN training.

The dataset is stored as individual PNG images in HDFS and loaded through Spark. The reconstructed data is then passed to PyTorch for preprocessing and distributed CNN training.

With DDP, each process maintains its own copy of the CNN and processes a different subset of the training data. Gradients are synchronised between processes during training.

The CNN architecture, preprocessing, data augmentation, class-imbalance handling, optimizer, number of epochs, and evaluation procedure are kept consistent with the local and Spark + HDFS experiments. This allows us to compare the effect of distributed training while keeping the underlying machine-learning experiment controlled.

In [36]:
import io
import os
import sys
import time
import random
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PySpark:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

Python: 3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]
PyTorch: 2.14.1+cu130
PySpark: 4.1.2
PySpark location: /home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py


### 1. Reproducibility

We keep it identical to the baseline.

In [4]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

g = torch.Generator()
g.manual_seed(seed)

### 2. Helper functions



In [28]:
# convert the PNG bytes to NumPy
def decode_image(image_bytes):
    return np.array(Image.open(io.BytesIO(image_bytes)).convert("RGB"))

def sort_by_original_index(rows):
    return sorted(
        rows,
        key=lambda row: int(row.path.rsplit("/", 1)[-1].replace(".png", ""))
    )

#### 2.1. Dataset class - PyTorch dataset adapter

The Spark pipeline produces the image data as NumPy arrays, whereas PyTorch's DataLoader expects a dataset implementing the standard Dataset interface. 

DermaMNISTArrayDataset class provides this interface by retrieving an image and label by index, converting the NumPy image to a PIL image for compatibility with torchvision transformations, applying the same preprocessing as the baseline, and returning the image and label as PyTorch tensors.

In [29]:
class DermaMNISTArrayDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # convert NumPy image back to PIL for torchvision transforms
        image = Image.fromarray(image)

        if self.transform is not None:
            image = self.transform(image)

        # keep label shape consistent with the baseline
        return image, torch.tensor([label], dtype=torch.long)

#### 2.2. CNN architecture definition

We use the same CNN architecture as in the baseline.

In [30]:
class CNN_deep(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

#### 2.3. Distributed Training Setup

The Spark/HDFS data-processing pipeline is completed before we initialize the distributed training.

PyTorch Distributed Data Parallel (DDP) is then initialized so that multiple CPU processes can train the CNN collaboratively. Each process maintains its own model replica and processes a separate portion of the training data, while gradients are synchronised across processes during backpropagation.

In [33]:
def setup_distributed():
    # read the process information provided by torchrun
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    # initialise communication between the processes
    dist.init_process_group(
        backend="gloo",   # because Gloo is for CPUs
        rank=rank,
        world_size=world_size
    )

    return rank, local_rank, world_size

def cleanup_distributed():
    # close the distributed process group after training
    dist.destroy_process_group()

#### 2.4. DDP-compatible Weighted Sampler

We need a small DDP-compatible replacement for `WeightedRandomSampler` that preserves exact square-root inverse-frequency weighting.

In [34]:
class DistributedWeightedSampler(torch.utils.data.Sampler):
    def __init__(self, weights, num_replicas, rank, seed=42):
        self.weights = weights
        self.num_replicas = num_replicas
        self.rank = rank
        self.seed = seed

        self.num_samples = len(weights) // num_replicas
        self.total_size = self.num_samples * num_replicas

        self.epoch = 0

    def __iter__(self):
        # generate one weighted sample list for the whole DDP job
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch)

        indices = torch.multinomial(self.weights, self.total_size, replacement=True, generator=g)

        # give each process its own subset
        indices = indices[self.rank:self.total_size:self.num_replicas]

        return iter(indices.tolist())

    def __len__(self):
        return self.num_samples

    def set_epoch(self, epoch):
        self.epoch = epoch

### 3. Main distributed experiment

In [39]:
def main():
    pipeline_start = time.perf_counter()
    
    # initialise the distributed environment
    rank, local_rank, world_size = setup_distributed()

    # only rank 0 prints the main experiment information
    if rank == 0:
        print(f"DDP world size: {world_size}")
        print("Backend: gloo")
        print("Device: CPU")

    # Spark + HDFS data pipeline
    # only rank 0 performs the Spark/HDFS data retrieval (this avoids starting one Spark application per DDP process)
    if rank == 0:
        # start Spark
        spark_start = time.perf_counter()

        spark = (
            SparkSession.builder
            .appName("DermaMNIST_Spark_HDFS_DDP")
            .master("local[*]")
            .getOrCreate()
        )

        spark_startup_time = time.perf_counter() - spark_start

        # read the image files from HDFS
        hdfs_dataset_path = "hdfs:///ca1/dermamnist_spark"
        
        spark_read_start = time.perf_counter()

        images_df = (spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(hdfs_dataset_path))

        # extract split and class from the HDFS path
        images_df = (
            images_df
            .withColumn(
                "split",
                regexp_extract("path", r"/(train|val|test)/", 1)
            )
            .withColumn(
                "class_id",
                regexp_extract("path", r"/(?:train|val|test)/(\d+)/", 1).cast("int")
            )
        )

        image_count = images_df.count()

        spark_read_time = time.perf_counter() - spark_read_start

        # transfer Spark data to Python
        spark_to_python_start = time.perf_counter()

        image_rows = (images_df.select("path", "content", "split", "class_id").collect())

        spark_to_python_time = (time.perf_counter() - spark_to_python_start)

        print(f"Records transferred: {len(image_rows)}")

        # convert PNG bytes to NumPy arrays
        conversion_start = time.perf_counter()

        train_rows = sort_by_original_index([row for row in image_rows if row.split == "train"])
        val_rows = sort_by_original_index([row for row in image_rows if row.split == "val"])
        test_rows = sort_by_original_index([row for row in image_rows if row.split == "test"])

        train_images = np.stack([decode_image(row.content) for row in train_rows])
        train_labels = np.array([row.class_id for row in train_rows], dtype=np.int64)

        val_images = np.stack([decode_image(row.content) for row in val_rows])
        val_labels = np.array([row.class_id for row in val_rows], dtype=np.int64)

        test_images = np.stack([decode_image(row.content) for row in test_rows])
        test_labels = np.array([row.class_id for row in test_rows], dtype=np.int64)

        conversion_time = time.perf_counter() - conversion_start

        print(f"Train: {train_images.shape} {train_labels.shape}")
        print(f"Val:   {val_images.shape} {val_labels.shape}")
        print(f"Test:  {test_images.shape} {test_labels.shape}")

        # calculate normalization statistics
        train_images_float = (train_images.astype(np.float32) / 255.0)

        mean_spark = train_images_float.mean(axis=(0, 1, 2))
        std_spark = train_images_float.std(axis=(0, 1, 2))

        print("Training mean:", mean_spark)
        print("Training std:", std_spark)

        # Spark is no longer needed after the HDFS data has been transferred to Python
        spark.stop()

    
    # broadcast dataset from rank 0 to all DDP processes
    # broadcast dataset sizes
    if rank == 0:
        dataset_sizes = torch.tensor(
            [len(train_images), len(val_images), len(test_images)], dtype=torch.long
        )
    else:
        dataset_sizes = torch.zeros(3, dtype=torch.long)

    dist.broadcast(dataset_sizes, src=0)
    train_size = dataset_sizes[0].item()
    val_size = dataset_sizes[1].item()
    test_size = dataset_sizes[2].item()

    # broadcast images and labels
    broadcast_start = time.perf_counter()
    
    if rank == 0:
        train_images_tensor = torch.from_numpy(train_images)
        train_labels_tensor = torch.from_numpy(train_labels)

        val_images_tensor = torch.from_numpy(val_images)
        val_labels_tensor = torch.from_numpy(val_labels)

        test_images_tensor = torch.from_numpy(test_images)
        test_labels_tensor = torch.from_numpy(test_labels)

        mean_tensor = torch.tensor(mean_spark, dtype=torch.float32)
        std_tensor = torch.tensor(std_spark, dtype=torch.float32)
    
    else:
        train_images_tensor = torch.empty((train_size, 64, 64, 3), dtype=torch.uint8)
        train_labels_tensor = torch.empty((train_size,), dtype=torch.long)

        val_images_tensor = torch.empty((val_size, 64, 64, 3), dtype=torch.uint8)
        val_labels_tensor = torch.empty((val_size,), dtype=torch.long)

        test_images_tensor = torch.empty((test_size, 64, 64, 3), dtype=torch.uint8)
        test_labels_tensor = torch.empty((test_size,), dtype=torch.long)

        mean_tensor = torch.empty(3, dtype=torch.float32)
        std_tensor = torch.empty(3, dtype=torch.float32)
  
    # broadcast the data from rank 0 to every other process
    dist.broadcast(train_images_tensor, src=0)
    dist.broadcast(train_labels_tensor, src=0)

    dist.broadcast(val_images_tensor, src=0)
    dist.broadcast(val_labels_tensor, src=0)

    dist.broadcast(test_images_tensor, src=0)
    dist.broadcast(test_labels_tensor, src=0)

    dist.broadcast(mean_tensor, src=0)
    dist.broadcast(std_tensor, src=0)

    broadcast_time = (time.perf_counter() - broadcast_start)

    # convert back to NumPy arrays for the Dataset class
    train_images = train_images_tensor.numpy()
    train_labels = train_labels_tensor.numpy()

    val_images = val_images_tensor.numpy()
    val_labels = val_labels_tensor.numpy()

    test_images = test_images_tensor.numpy()
    test_labels = test_labels_tensor.numpy()

    mean_spark = mean_tensor.numpy()
    std_spark = std_tensor.numpy()

    # make sure every process has finished receiving the data
    dist.barrier()

    # prepare the data for PyTorch
    pytorch_preparation_start = time.perf_counter()

    # same transforms as the baseline
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean_spark.tolist(), std=std_spark.tolist())
    ])

    transform_eval = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean_spark.tolist(), std=std_spark.tolist())
    ])

    train_dataset = DermaMNISTArrayDataset(train_images, train_labels, transform=transform_train)
    val_dataset = DermaMNISTArrayDataset(val_images, val_labels, transform=transform_eval)
    test_dataset = DermaMNISTArrayDataset(test_images, test_labels, transform=transform_eval)

    # create DDP-compatible training sampler
    class_counts = np.bincount(train_labels)

    class_sampling_weights = 1 / np.sqrt(class_counts)

    sample_weights = class_sampling_weights[train_labels]
    sample_weights = torch.tensor(sample_weights, dtype=torch.double)

    train_sampler = DistributedWeightedSampler(
        weights=sample_weights,
        num_replicas=world_size,
        rank=rank,
        seed=seed
    )

    batch_size = 64

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    pytorch_preparation_time = (time.perf_counter() - pytorch_preparation_start)

    # initialise the model
    model_start = time.perf_counter()

    model = CNN_deep(num_classes=7)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # wrap the model with DistributedDataParallel
    model = DDP(model)

    model_initialization_time = (time.perf_counter() - model_start)


    # distributed training
    num_epochs = 50

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_macro_f1": [],
        "epoch_time": []
    }

    best_macro_f1 = -1
    best_epoch = 0
    best_model_state = None

    # synchronize before starting the measured training phase
    dist.barrier()

    training_start = time.perf_counter()

    for epoch in range(num_epochs):
        # make sure all processes start the epoch together
        dist.barrier()
        
        epoch_start = time.perf_counter()

        # change the sampler's shuffle order for this epoch
        train_sampler.set_epoch(epoch)

        # training
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0
    
        for images, labels in train_loader:
            labels = labels.squeeze(1)
            
            optimizer.zero_grad()   # clear gradients from the previous batch
    
            outputs = model(images)   # perform the forward pass
            loss = criterion(outputs, labels)   # calculate the classification loss
            loss.backward()   # calculate gradients
            optimizer.step()   # update model parameters

            # accumulate training statistics
            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
        # aggregate training metrics across DDP processes
        train_metrics = torch.tensor([running_loss, correct, total], dtype=torch.float64)
        dist.all_reduce(train_metrics, op=dist.ReduceOp.SUM)
        train_loss = train_metrics[0].item() / train_metrics[2].item()
        train_acc = train_metrics[1].item() / train_metrics[2].item()

        # validation
        model.eval()
    
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        all_val_predictions = []
        all_val_labels = []
    
        with torch.no_grad():
            for images, labels in val_loader:
                labels = labels.squeeze(1)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_running_loss += (loss.item() * images.size(0))
                predictions = outputs.argmax(dim=1)
                val_correct += ((predictions == labels).sum().item())
                val_total += labels.size(0)

                all_val_predictions.extend(predictions.numpy())
                all_val_labels.extend(labels.numpy())

        val_loss = (val_running_loss / val_total)
        val_acc = (val_correct / val_total)
        val_macro_f1 = f1_score(all_val_labels, all_val_predictions, average="macro", zero_division=0)
        
        epoch_time = time.perf_counter() - epoch_start

        # only rank 0 records and prints the results
        if rank == 0:

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["val_macro_f1"].append(val_macro_f1)
            history["epoch_time"].append(epoch_time)

            if val_macro_f1 > best_macro_f1:
                best_macro_f1 = val_macro_f1
                best_epoch = epoch + 1
                best_model_state = copy.deepcopy(model.module.state_dict())
            
            print(
                f"Epoch {epoch + 1:2d}/{num_epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:.4f} | "
                f"Val Macro-F1: {val_macro_f1:.4f} | "
                f"Time: {epoch_time:.2f}s"
            )

    training_time = time.perf_counter() - training_start

    if rank == 0:

        print(f"\nTraining time: {training_time:.2f}s")
        print(f"Best validation macro-F1: {best_macro_f1:.4f}")
        print(f"Best epoch: {best_epoch}")

    
    # restore best model and evaluate test set
    dist.barrier()
    
    # only rank 0 performs the final test evaluation
    if rank == 0:
        # restore the best checkpoint kept in memory
        model.module.load_state_dict(best_model_state)
        
        test_start = time.perf_counter()

        model.eval()

        test_running_loss = 0.0
        test_correct = 0
        test_total = 0

        all_test_labels = []
        all_test_predictions = []

        with torch.no_grad():
            for images, labels in test_loader:
                labels = labels.squeeze(1)

                outputs = model(images)
                loss = criterion(outputs, labels)
                test_running_loss += (loss.item() * images.size(0))
                predictions = outputs.argmax(dim=1)
                test_correct += ((predictions == labels).sum().item())
                test_total += labels.size(0)
                all_test_labels.extend(labels.numpy())
                all_test_predictions.extend(predictions.numpy())

        test_time = (time.perf_counter() - test_start)

        test_loss = (test_running_loss / test_total)
        test_acc = (test_correct / test_total)
        test_macro_f1 = f1_score(all_test_labels, all_test_predictions, average="macro", zero_division=0)

        print(f"Test evaluation time: {test_time:.2f}s")
        print(f"Test loss: {test_loss:.4f}")
        print(f"Test accuracy: {test_acc:.4f}")
        print(f"Test macro-F1: {test_macro_f1:.4f}")

        # classification report
        print()
        print("\nClassification report:")
        print(
            classification_report(all_test_labels, all_test_predictions, labels=list(range(7)), zero_division=0)
        )

        # confusion matrix
        cm = confusion_matrix(all_test_labels, all_test_predictions, labels=list(range(7)))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(7)))

        fig, ax = plt.subplots(figsize=(7, 7))
        disp.plot(ax=ax, cmap="Blues", values_format="d")
        ax.set_title("DermaMNIST DDP Confusion Matrix")
        fig.tight_layout()
        fig.savefig("dermamnist_ddp_confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

        print("Confusion matrix saved to: dermamnist_ddp_confusion_matrix.png")

    else:
        test_time = 0.0

    # make sure rank 0 finishes testing before any process destroys the process group
    dist.barrier()

    # final timing summary
    if rank == 0:

        average_epoch_time = np.mean(history["epoch_time"])

        # sum the measured pipeline stages
        measured_pipeline_time = (spark_startup_time + spark_read_time + spark_to_python_time + conversion_time
            + broadcast_time + pytorch_preparation_time + model_initialization_time + training_time + test_time)

        print()
        print(f"Spark startup: {spark_startup_time:.2f}s")
        print(f"Spark data read: {spark_read_time:.2f}s")
        print(f"Spark → Python transfer: {spark_to_python_time:.2f}s")
        print(f"Image conversion: {conversion_time:.2f}s")
        print(f"DDP data broadcast: {broadcast_time:.2f}s")
        print(f"PyTorch data preparation: {pytorch_preparation_time:.2f}s")
        print(f"Model initialization: {model_initialization_time:.2f}s")
        print(f"Training: {training_time:.2f}s")
        print(f"Average epoch: {average_epoch_time:.2f}s")
        print(f"Test evaluation: {test_time:.2f}s")
        print()
        print(f"Measured pipeline time: {measured_pipeline_time:.2f}s")

    # finish the distributed training session
    cleanup_distributed()

### 4. Launch the training

In [ ]:
if __name__ == "__main__":
    main()

## Actual execution is in terminal

The next step is to execute this notebook through `nbconvert` and launch the resulting Python process with `torchrun --standalone --nproc-per-node=2 my_script.py`

There are two separate concepts here:
- number of DDP processes → `--nproc-per-node`
- number of CPU threads available to each process → controlled separately by PyTorch/environment

For our first test, we use 2 DDP processes on our VM's 8 CPUs.

So:

8 VM CPUs
   ↓
2 DDP processes
   ↓
each process can use multiple CPU threads


Since our file is `03_DermaMNIST_Spark_HDFS_DDP.ipynb`, we first need to convert it to a temporary Python script with `jupyter nbconvert --to script 03_MNIST_DDP.ipynb`. This should create `03_DermaMNIST_Spark_HDFS_DDP.py`.

Then, we launch **2 DDP processes** with `torchrun --standalone --nproc-per-node=2 03_DermaMNIST_Spark_HDFS_DDP.py`

### Terminal output with 2 processes:
 
W1005 19:11:39.817000 217407 site-packages/torch/distributed/run.py:982] *****************************************

W1005 19:11:39.817000 217407 site-packages/torch/distributed/run.py:982] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 

W1005 19:11:39.817000 217407 site-packages/torch/distributed/run.py:982] *****************************************

Python: 3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]

PyTorch: 2.14.1+cu130

PySpark: 4.1.2

PySpark location: /home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py

Python: 3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]

PyTorch: 2.14.1+cu130

PySpark: 4.1.2

PySpark location: /home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py

DDP world size: 2

Backend: gloo

Device: CPU

WARNING: Using incubator modules: jdk.incubator.vector

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties

Setting default log level to "WARN".

To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

26/10/05 19:11:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

26/10/05 19:11:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.

26/10/05 19:11:45 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.

26/10/05 19:11:45 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.

Records transferred: 10015                                                      

Train: (7007, 64, 64, 3) (7007,)

Val:   (1003, 64, 64, 3) (1003,)

Test:  (2005, 64, 64, 3) (2005,)

Training mean: [0.5845583  0.58314306 0.5845583 ]

Training std: [0.21778087 0.15745011 0.16998562]

Epoch  1/50 | Train Loss: 1.6449 | Train Acc: 0.3837 | Val Loss: 0.9787 | Val Acc: 0.6889 | Val Macro-F1: 0.2349 | Time: 29.43s

Epoch  2/50 | Train Loss: 1.3803 | Train Acc: 0.4615 | Val Loss: 0.8945 | Val Acc: 0.6999 | Val Macro-F1: 0.3538 | Time: 29.93s

Epoch  3/50 | Train Loss: 1.2596 | Train Acc: 0.5130 | Val Loss: 0.7896 | Val Acc: 0.7168 | Val Macro-F1: 0.3601 | Time: 30.13s


Epoch  4/50 | Train Loss: 1.1708 | Train Acc: 0.5435 | Val Loss: 0.8053 | Val Acc: 0.6640 | Val Macro-F1: 0.3686 | Time: 29.23s

Epoch  5/50 | Train Loss: 1.1457 | Train Acc: 0.5484 | Val Loss: 0.7565 | Val Acc: 0.6999 | Val Macro-F1: 0.4710 | Time: 29.01s

Epoch  6/50 | Train Loss: 1.0591 | Train Acc: 0.5862 | Val Loss: 0.7607 | Val Acc: 0.6949 | Val Macro-F1: 0.4849 | Time: 28.74s

Epoch  7/50 | Train Loss: 1.0127 | Train Acc: 0.6013 | Val Loss: 0.7699 | Val Acc: 0.7109 | Val Macro-F1: 0.4922 | Time: 28.61s

Epoch  8/50 | Train Loss: 0.9732 | Train Acc: 0.6152 | Val Loss: 0.7790 | Val Acc: 0.6919 | Val Macro-F1: 0.4663 | Time: 28.76s

Epoch  9/50 | Train Loss: 0.9683 | Train Acc: 0.6223 | Val Loss: 0.6778 | Val Acc: 0.7498 | Val Macro-F1: 0.5230 | Time: 28.69s

Epoch 10/50 | Train Loss: 0.9253 | Train Acc: 0.6302 | Val Loss: 0.6754 | Val Acc: 0.7248 | Val Macro-F1: 0.5037 | Time: 28.63s

Epoch 11/50 | Train Loss: 0.8961 | Train Acc: 0.6524 | Val Loss: 0.6899 | Val Acc: 0.7188 | Val Macro-F1: 0.5078 | Time: 28.66s

Epoch 12/50 | Train Loss: 0.8588 | Train Acc: 0.6606 | Val Loss: 0.7247 | Val Acc: 0.7019 | Val Macro-F1: 0.5294 | Time: 28.58s

Epoch 13/50 | Train Loss: 0.8394 | Train Acc: 0.6647 | Val Loss: 0.6889 | Val Acc: 0.7039 | Val Macro-F1: 0.5304 | Time: 28.68s

Epoch 14/50 | Train Loss: 0.8160 | Train Acc: 0.6709 | Val Loss: 0.7183 | Val Acc: 0.7149 | Val Macro-F1: 0.5174 | Time: 28.67s

Epoch 15/50 | Train Loss: 0.7894 | Train Acc: 0.6933 | Val Loss: 0.6733 | Val Acc: 0.7438 | Val Macro-F1: 0.5216 | Time: 28.63s

Epoch 16/50 | Train Loss: 0.7573 | Train Acc: 0.7122 | Val Loss: 0.6911 | Val Acc: 0.7258 | Val Macro-F1: 0.5209 | Time: 29.57s

Epoch 17/50 | Train Loss: 0.7397 | Train Acc: 0.7098 | Val Loss: 0.7017 | Val Acc: 0.7238 | Val Macro-F1: 0.5436 | Time: 29.02s

Epoch 18/50 | Train Loss: 0.7130 | Train Acc: 0.7204 | Val Loss: 0.6724 | Val Acc: 0.7438 | Val Macro-F1: 0.5473 | Time: 30.08s

Epoch 19/50 | Train Loss: 0.6787 | Train Acc: 0.7359 | Val Loss: 0.6376 | Val Acc: 0.7727 | Val Macro-F1: 0.5851 | Time: 28.76s

Epoch 20/50 | Train Loss: 0.6296 | Train Acc: 0.7564 | Val Loss: 0.7054 | Val Acc: 0.7348 | Val Macro-F1: 0.5585 | Time: 28.79s

Epoch 21/50 | Train Loss: 0.6272 | Train Acc: 0.7582 | Val Loss: 0.6720 | Val Acc: 0.7557 | Val Macro-F1: 0.5579 | Time: 28.79s

Epoch 22/50 | Train Loss: 0.6022 | Train Acc: 0.7652 | Val Loss: 0.7096 | Val Acc: 0.7348 | Val Macro-F1: 0.5504 | Time: 28.70s

Epoch 23/50 | Train Loss: 0.5750 | Train Acc: 0.7662 | Val Loss: 0.8223 | Val Acc: 0.6471 | Val Macro-F1: 0.4987 | Time: 28.70s

Epoch 24/50 | Train Loss: 0.5760 | Train Acc: 0.7698 | Val Loss: 0.7402 | Val Acc: 0.7218 | Val Macro-F1: 0.5573 | Time: 29.08s

Epoch 25/50 | Train Loss: 0.5396 | Train Acc: 0.7832 | Val Loss: 0.7218 | Val Acc: 0.7647 | Val Macro-F1: 0.5583 | Time: 29.21s

Epoch 26/50 | Train Loss: 0.4978 | Train Acc: 0.8100 | Val Loss: 0.7147 | Val Acc: 0.7308 | Val Macro-F1: 0.5685 | Time: 28.86s

Epoch 27/50 | Train Loss: 0.5185 | Train Acc: 0.7986 | Val Loss: 0.8043 | Val Acc: 0.7139 | Val Macro-F1: 0.5567 | Time: 28.95s

Epoch 28/50 | Train Loss: 0.5202 | Train Acc: 0.7949 | Val Loss: 0.7166 | Val Acc: 0.7517 | Val Macro-F1: 0.5625 | Time: 28.65s

Epoch 29/50 | Train Loss: 0.4717 | Train Acc: 0.8173 | Val Loss: 0.7683 | Val Acc: 0.7238 | Val Macro-F1: 0.5726 | Time: 28.59s

Epoch 30/50 | Train Loss: 0.4323 | Train Acc: 0.8289 | Val Loss: 0.7431 | Val Acc: 0.7577 | Val Macro-F1: 0.5704 | Time: 28.50s

Epoch 31/50 | Train Loss: 0.4174 | Train Acc: 0.8316 | Val Loss: 0.7686 | Val Acc: 0.7507 | Val Macro-F1: 0.5669 | Time: 28.49s

Epoch 32/50 | Train Loss: 0.4022 | Train Acc: 0.8391 | Val Loss: 0.7872 | Val Acc: 0.7647 | Val Macro-F1: 0.5784 | Time: 28.60s

Epoch 33/50 | Train Loss: 0.3634 | Train Acc: 0.8511 | Val Loss: 0.8684 | Val Acc: 0.7607 | Val Macro-F1: 0.5853 | Time: 28.50s

Epoch 34/50 | Train Loss: 0.3879 | Train Acc: 0.8467 | Val Loss: 0.8656 | Val Acc: 0.7358 | Val Macro-F1: 0.5747 | Time: 30.49s

Epoch 35/50 | Train Loss: 0.3877 | Train Acc: 0.8424 | Val Loss: 0.8015 | Val Acc: 0.7547 | Val Macro-F1: 0.5637 | Time: 28.71s

Epoch 36/50 | Train Loss: 0.3583 | Train Acc: 0.8540 | Val Loss: 0.8519 | Val Acc: 0.7647 | Val Macro-F1: 0.5924 | Time: 29.91s

Epoch 37/50 | Train Loss: 0.3408 | Train Acc: 0.8638 | Val Loss: 0.8657 | Val Acc: 0.7747 | Val Macro-F1: 0.5898 | Time: 28.64s

Epoch 38/50 | Train Loss: 0.3177 | Train Acc: 0.8728 | Val Loss: 0.9414 | Val Acc: 0.7617 | Val Macro-F1: 0.5911 | Time: 28.74s

Epoch 39/50 | Train Loss: 0.3232 | Train Acc: 0.8775 | Val Loss: 0.9415 | Val Acc: 0.7757 | Val Macro-F1: 0.6248 | Time: 28.55s

Epoch 40/50 | Train Loss: 0.3006 | Train Acc: 0.8797 | Val Loss: 0.9487 | Val Acc: 0.7817 | Val Macro-F1: 0.6029 | Time: 28.63s

Epoch 41/50 | Train Loss: 0.3127 | Train Acc: 0.8782 | Val Loss: 0.9725 | Val Acc: 0.7498 | Val Macro-F1: 0.5779 | Time: 28.75s

Epoch 42/50 | Train Loss: 0.2511 | Train Acc: 0.9034 | Val Loss: 1.0795 | Val Acc: 0.7587 | Val Macro-F1: 0.5985 | Time: 28.87s

Epoch 43/50 | Train Loss: 0.2691 | Train Acc: 0.8947 | Val Loss: 0.9790 | Val Acc: 0.7767 | Val Macro-F1: 0.5967 | Time: 28.64s

Epoch 44/50 | Train Loss: 0.2833 | Train Acc: 0.8907 | Val Loss: 1.0248 | Val Acc: 0.7418 | Val Macro-F1: 0.5786 | Time: 28.55s

Epoch 45/50 | Train Loss: 0.2492 | Train Acc: 0.9051 | Val Loss: 1.0055 | Val Acc: 0.7687 | Val Macro-F1: 0.6037 | Time: 28.68s

Epoch 46/50 | Train Loss: 0.2451 | Train Acc: 0.9037 | Val Loss: 0.9882 | Val Acc: 0.7637 | Val Macro-F1: 0.5992 | Time: 28.51s

Epoch 47/50 | Train Loss: 0.2475 | Train Acc: 0.8997 | Val Loss: 1.0746 | Val Acc: 0.7567 | Val Macro-F1: 0.6049 | Time: 28.40s

Epoch 48/50 | Train Loss: 0.2263 | Train Acc: 0.9132 | Val Loss: 1.1659 | Val Acc: 0.7747 | Val Macro-F1: 0.5979 | Time: 28.46s

Epoch 49/50 | Train Loss: 0.2331 | Train Acc: 0.9088 | Val Loss: 1.0595 | Val Acc: 0.7657 | Val Macro-F1: 0.6031 | Time: 28.54s

Epoch 50/50 | Train Loss: 0.2118 | Train Acc: 0.9171 | Val Loss: 1.1367 | Val Acc: 0.7737 | Val Macro-F1: 0.6029 | Time: 28.42s

Training time: 1443.79s

Best validation macro-F1: 0.6248

Best epoch: 39

Test evaluation time: 4.02s

Test loss: 0.8837

Test accuracy: 0.7661

Test macro-F1: 0.6137


Classification report:
              precision    recall  f1-score   support

           0       0.53      0.35      0.42        66
           1       0.52      0.63      0.57       103
           2       0.59      0.58      0.58       220
           3       0.59      0.43      0.50        23
           4       0.45      0.57      0.50       223
           5       0.90      0.87      0.88      1341
           6       0.88      0.79      0.84        29

    accuracy                           0.77      2005
    macro avg       0.64      0.60     0.61      2005
    weighted avg    0.78      0.77     0.77      2005

Confusion matrix saved to: dermamnist_ddp_confusion_matrix.png

Spark startup: 3.80s

Spark data read: 4.54s

Spark → Python transfer: 7.83s

Image conversion: 1.89s

DDP data broadcast: 0.04s

PyTorch data preparation: 0.00s

Model initialization: 0.02s

Training: 1443.79s

Average epoch: 28.87s

Test evaluation: 4.02s

Measured pipeline time: 1465.94s

### Terminal output with 4 processes:

W1005 19:57:39.495000 218662 site-packages/torch/distributed/run.py:982] *****************************************

W1005 19:57:39.495000 218662 site-packages/torch/distributed/run.py:982] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 

W1005 19:57:39.495000 218662 site-packages/torch/distributed/run.py:982] *****************************************

Python:Python:Python: 3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]  

3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]

PyTorch: PyTorch: PyTorch:2.14.1+cu1302.14.1+cu130 

2.14.1+cu130PySpark:
PySpark:  PySpark:4.1.24.1.2

 PySpark location:PySpark location:4.1.2 
 Python:/home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.pyPySpark location:/home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py
  
3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]/home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py

PyTorch: 2.14.1+cu130

PySpark: 4.1.2

PySpark location: /home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py

DDP world size: 4

Backend: gloo

Device: CPU

WARNING: Using incubator modules: jdk.incubator.vector

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/05 19:57:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

Records transferred: 10015                                                      

Train: (7007, 64, 64, 3) (7007,)

Val:   (1003, 64, 64, 3) (1003,)

Test:  (2005, 64, 64, 3) (2005,)

Training mean: [0.5845583  0.58314306 0.5845583 ]

Training std: [0.21778087 0.15745011 0.16998562]

Epoch  1/50 | Train Loss: 1.7341 | Train Acc: 0.3549 | Val Loss: 1.2994 | Val Acc: 0.6690 | Val Macro-F1: 0.1145 | Time: 17.27s

Epoch  2/50 | Train Loss: 1.5339 | Train Acc: 0.3992 | Val Loss: 1.0720 | Val Acc: 0.6810 | Val Macro-F1: 0.2287 | Time: 17.06s

Epoch  3/50 | Train Loss: 1.3958 | Train Acc: 0.4549 | Val Loss: 0.8557 | Val Acc: 0.7019 | Val Macro-F1: 0.2633 | Time: 17.25s

Epoch  4/50 | Train Loss: 1.2796 | Train Acc: 0.5079 | Val Loss: 0.9187 | Val Acc: 0.6171 | Val Macro-F1: 0.3192 | Time: 17.44s

Epoch  5/50 | Train Loss: 1.2162 | Train Acc: 0.5214 | Val Loss: 0.7784 | Val Acc: 0.7079 | Val Macro-F1: 0.4071 | Time: 17.78s

Epoch  6/50 | Train Loss: 1.1410 | Train Acc: 0.5581 | Val Loss: 0.8233 | Val Acc: 0.6780 | Val Macro-F1: 0.4533 | Time: 17.81s

Epoch  7/50 | Train Loss: 1.0781 | Train Acc: 0.5835 | Val Loss: 0.7529 | Val Acc: 0.7109 | Val Macro-F1: 0.4610 | Time: 17.28s

Epoch  8/50 | Train Loss: 1.0700 | Train Acc: 0.5868 | Val Loss: 0.7691 | Val Acc: 0.6620 | Val Macro-F1: 0.4640 | Time: 17.11s

Epoch  9/50 | Train Loss: 1.0340 | Train Acc: 0.5878 | Val Loss: 0.7359 | Val Acc: 0.7328 | Val Macro-F1: 0.4502 | Time: 17.17s

Epoch 10/50 | Train Loss: 1.0197 | Train Acc: 0.5981 | Val Loss: 0.7776 | Val Acc: 0.7009 | Val Macro-F1: 0.4798 | Time: 17.64s

Epoch 11/50 | Train Loss: 0.9886 | Train Acc: 0.6107 | Val Loss: 0.7714 | Val Acc: 0.6610 | Val Macro-F1: 0.4494 | Time: 17.10s

Epoch 12/50 | Train Loss: 0.9449 | Train Acc: 0.6352 | Val Loss: 0.6947 | Val Acc: 0.7079 | Val Macro-F1: 0.4985 | Time: 17.12s

Epoch 13/50 | Train Loss: 0.9721 | Train Acc: 0.6124 | Val Loss: 0.7656 | Val Acc: 0.6929 | Val Macro-F1: 0.4485 | Time: 17.13s

Epoch 14/50 | Train Loss: 0.9194 | Train Acc: 0.6361 | Val Loss: 0.7337 | Val Acc: 0.6849 | Val Macro-F1: 0.5003 | Time: 17.12s

Epoch 15/50 | Train Loss: 0.9092 | Train Acc: 0.6409 | Val Loss: 0.7126 | Val Acc: 0.7278 | Val Macro-F1: 0.4871 | Time: 17.11s

Epoch 16/50 | Train Loss: 0.8834 | Train Acc: 0.6551 | Val Loss: 0.8231 | Val Acc: 0.6341 | Val Macro-F1: 0.4912 | Time: 17.13s

Epoch 17/50 | Train Loss: 0.8964 | Train Acc: 0.6516 | Val Loss: 0.6690 | Val Acc: 0.7428 | Val Macro-F1: 0.5129 | Time: 17.10s

Epoch 18/50 | Train Loss: 0.8604 | Train Acc: 0.6650 | Val Loss: 0.6596 | Val Acc: 0.7537 | Val Macro-F1: 0.5548 | Time: 17.31s

Epoch 19/50 | Train Loss: 0.8419 | Train Acc: 0.6685 | Val Loss: 0.6761 | Val Acc: 0.7408 | Val Macro-F1: 0.5405 | Time: 17.26s

Epoch 20/50 | Train Loss: 0.7795 | Train Acc: 0.7022 | Val Loss: 0.7209 | Val Acc: 0.6780 | Val Macro-F1: 0.5043 | Time: 17.10s

Epoch 21/50 | Train Loss: 0.7889 | Train Acc: 0.6940 | Val Loss: 0.6885 | Val Acc: 0.7198 | Val Macro-F1: 0.5352 | Time: 17.15s

Epoch 22/50 | Train Loss: 0.7625 | Train Acc: 0.7003 | Val Loss: 0.7644 | Val Acc: 0.6700 | Val Macro-F1: 0.5232 | Time: 17.10s

Epoch 23/50 | Train Loss: 0.7581 | Train Acc: 0.7050 | Val Loss: 0.6595 | Val Acc: 0.7448 | Val Macro-F1: 0.5442 | Time: 17.22s

Epoch 24/50 | Train Loss: 0.7258 | Train Acc: 0.7169 | Val Loss: 0.6597 | Val Acc: 0.7438 | Val Macro-F1: 0.5345 | Time: 17.07s

Epoch 25/50 | Train Loss: 0.7170 | Train Acc: 0.7257 | Val Loss: 0.6449 | Val Acc: 0.7567 | Val Macro-F1: 0.5473 | Time: 16.95s

Epoch 26/50 | Train Loss: 0.6939 | Train Acc: 0.7292 | Val Loss: 0.6950 | Val Acc: 0.7149 | Val Macro-F1: 0.5423 | Time: 16.91s

Epoch 27/50 | Train Loss: 0.6591 | Train Acc: 0.7453 | Val Loss: 0.6598 | Val Acc: 0.7577 | Val Macro-F1: 0.5647 | Time: 16.85s

Epoch 28/50 | Train Loss: 0.6585 | Train Acc: 0.7416 | Val Loss: 0.6306 | Val Acc: 0.7866 | Val Macro-F1: 0.6064 | Time: 16.92s

Epoch 29/50 | Train Loss: 0.6320 | Train Acc: 0.7490 | Val Loss: 0.6626 | Val Acc: 0.7667 | Val Macro-F1: 0.6100 | Time: 16.91s

Epoch 30/50 | Train Loss: 0.5913 | Train Acc: 0.7683 | Val Loss: 0.6955 | Val Acc: 0.7517 | Val Macro-F1: 0.5720 | Time: 16.91s

Epoch 31/50 | Train Loss: 0.5738 | Train Acc: 0.7766 | Val Loss: 0.6756 | Val Acc: 0.7846 | Val Macro-F1: 0.5687 | Time: 16.91s

Epoch 32/50 | Train Loss: 0.5986 | Train Acc: 0.7644 | Val Loss: 0.7321 | Val Acc: 0.7208 | Val Macro-F1: 0.5560 | Time: 16.91s

Epoch 33/50 | Train Loss: 0.5473 | Train Acc: 0.7863 | Val Loss: 0.6874 | Val Acc: 0.7378 | Val Macro-F1: 0.5677 | Time: 17.30s

Epoch 34/50 | Train Loss: 0.5311 | Train Acc: 0.7927 | Val Loss: 0.6864 | Val Acc: 0.7408 | Val Macro-F1: 0.5648 | Time: 17.53s

Epoch 35/50 | Train Loss: 0.5344 | Train Acc: 0.7923 | Val Loss: 0.7295 | Val Acc: 0.7268 | Val Macro-F1: 0.5765 | Time: 16.95s

Epoch 36/50 | Train Loss: 0.5015 | Train Acc: 0.7983 | Val Loss: 0.7324 | Val Acc: 0.7328 | Val Macro-F1: 0.5880 | Time: 16.93s

Epoch 37/50 | Train Loss: 0.4901 | Train Acc: 0.8084 | Val Loss: 0.7473 | Val Acc: 0.7288 | Val Macro-F1: 0.5689 | Time: 16.94s

Epoch 38/50 | Train Loss: 0.4746 | Train Acc: 0.8098 | Val Loss: 0.7379 | Val Acc: 0.7438 | Val Macro-F1: 0.5836 | Time: 17.09s

Epoch 39/50 | Train Loss: 0.4627 | Train Acc: 0.8141 | Val Loss: 0.7465 | Val Acc: 0.7527 | Val Macro-F1: 0.5980 | Time: 17.00s

Epoch 40/50 | Train Loss: 0.4467 | Train Acc: 0.8267 | Val Loss: 0.7637 | Val Acc: 0.7258 | Val Macro-F1: 0.5413 | Time: 17.06s

Epoch 41/50 | Train Loss: 0.4349 | Train Acc: 0.8254 | Val Loss: 0.7531 | Val Acc: 0.7557 | Val Macro-F1: 0.6060 | Time: 16.98s

Epoch 42/50 | Train Loss: 0.3962 | Train Acc: 0.8412 | Val Loss: 0.7710 | Val Acc: 0.7288 | Val Macro-F1: 0.5723 | Time: 17.06s

Epoch 43/50 | Train Loss: 0.3811 | Train Acc: 0.8521 | Val Loss: 0.8127 | Val Acc: 0.7677 | Val Macro-F1: 0.5723 | Time: 17.02s

Epoch 44/50 | Train Loss: 0.3805 | Train Acc: 0.8479 | Val Loss: 0.8468 | Val Acc: 0.7757 | Val Macro-F1: 0.6132 | Time: 16.93s

Epoch 45/50 | Train Loss: 0.3933 | Train Acc: 0.8469 | Val Loss: 0.7935 | Val Acc: 0.7607 | Val Macro-F1: 0.5910 | Time: 17.09s

Epoch 46/50 | Train Loss: 0.3485 | Train Acc: 0.8675 | Val Loss: 0.7755 | Val Acc: 0.7617 | Val Macro-F1: 0.6100 | Time: 16.85s

Epoch 47/50 | Train Loss: 0.3369 | Train Acc: 0.8625 | Val Loss: 0.8795 | Val Acc: 0.7627 | Val Macro-F1: 0.5980 | Time: 16.93s

Epoch 48/50 | Train Loss: 0.3368 | Train Acc: 0.8676 | Val Loss: 0.9040 | Val Acc: 0.7617 | Val Macro-F1: 0.5780 | Time: 16.90s

Epoch 49/50 | Train Loss: 0.3475 | Train Acc: 0.8654 | Val Loss: 0.8537 | Val Acc: 0.7607 | Val Macro-F1: 0.5920 | Time: 17.39s

Epoch 50/50 | Train Loss: 0.3155 | Train Acc: 0.8782 | Val Loss: 0.9698 | Val Acc: 0.7378 | Val Macro-F1: 0.5789 | Time: 17.78s


Training time: 857.55s

Best validation macro-F1: 0.6132

Best epoch: 44

Test evaluation time: 4.17s

Test loss: 0.8523

Test accuracy: 0.7581

Test macro-F1: 0.5873


Classification report:
              precision    recall  f1-score   support

           0       0.58      0.42      0.49        66
           1       0.46      0.69      0.55       103
           2       0.64      0.41      0.50       220
           3       0.56      0.39      0.46        23
           4       0.43      0.54      0.48       223
           5       0.88      0.88      0.88      1341
           6       0.73      0.76      0.75        29

    accuracy                           0.76      2005
    macro avg       0.61      0.58     0.59      2005
    weighted avg    0.77      0.76     0.76      2005

Confusion matrix saved to: dermamnist_ddp_confusion_matrix.png

Spark startup: 3.16s

Spark data read: 4.11s

Spark → Python transfer: 5.24s

Image conversion: 1.33s

DDP data broadcast: 0.05s

PyTorch data preparation: 0.00s

Model initialization: 0.02s

Training: 857.55s

Average epoch: 17.14s

Test evaluation: 4.17s

Measured pipeline time: 875.62s

## HDFS + Spark + DDP pipeline conclusion

The distributed CNN experiments demonstrated that PyTorch Distributed Data Parallel (DDP) can reduce training time when the workload is distributed across multiple CPU processes. The DDP implementation reproduced the same DermaMNIST CNN architecture, preprocessing, weighted sampling strategy, and 50-epoch training procedure used in the local and Spark/HDFS experiments, while using Gloo for inter-process communication.

Increasing the number of DDP processes reduced the training time from **1560.53 s** for the single-process baseline to **1443.79 s** with 2 processes and **857.55 s** with 4 processes. This corresponds to a **1.82× training speedup** with 4 processes, although the speedup was sub-linear due to distributed communication and CPU resource overhead. The measured end-to-end pipeline time was reduced from **1564.58 s** for the baseline to **875.62 s** with 4 DDP processes.

The classification performance remained broadly comparable across the experiments, with test accuracy around 76%. The test macro-F1 scores were **0.6198** for the baseline, **0.6137** with 2 DDP processes, and **0.5873** with 4 DDP processes. These differences reflect variation in the distributed training process rather than a change to the CNN architecture or preprocessing pipeline.

Overall, the experiment shows that DDP can provide meaningful training-time improvements for this workload, while also demonstrating that increasing the number of processes does not result in linear speedup because of the overhead associated with distributed CPU training.